# Retail Credit Default Early Warning

## Phase 2: interactive exploration

This Snowflake Workspace notebook investigates the synthetic account portfolio created in Phase 1. The target is whether an existing account reaches the synthetic default state within 90 days.

The intended use is portfolio monitoring and case prioritisation. It is not an automated credit-approval, limit-setting, or adverse-action system. Protected characteristics and direct proxies are intentionally absent.

**Prerequisite:** run the rendered Phase 1 bootstrap and continue only after `verify_data.sql` returns `PHASE_1_STATUS = 'PASS'`.

## 1. Confirm the Workspace execution context

Before running the notebook, select `CRISK_DEMO_DEVELOPER` and `CRISK_DEMO_WH` with the Workspace role and warehouse pickers. The data queries use fully qualified relations, so they do not depend on an implicit database or schema. Run the following cell to confirm the active context.

In [ ]:
%%sql -r execution_context
SELECT
  CURRENT_ROLE() AS ACTIVE_ROLE,
  CURRENT_WAREHOUSE() AS ACTIVE_WAREHOUSE,
  CURRENT_DATABASE() AS ACTIVE_DATABASE,
  CURRENT_SCHEMA() AS ACTIVE_SCHEMA;

## 2. Inspect the data contract before choosing features

The training relation has one row per account and observation month with finalised ground truth. Start by examining its schema and a bounded sample rather than assuming which columns should enter a model.

In [ ]:
%%sql -r training_schema
DESCRIBE VIEW CRISK_DEMO_DB.RAW.TRAINING_BASE;

In [ ]:
%%sql -r training_sample
SELECT *
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
ORDER BY OBSERVATION_DATE, ACCOUNT_ID
LIMIT 20;

The sample should show operational account attributes, behavioural measures, a finalised 90-day outcome, and its finality date. Account ID, observation date, and outcome finality date describe identity or timing; they are not automatically modelling features.

In [ ]:
%%sql -r contract_summary
SELECT
  COUNT(*) AS TRAINING_ROW_COUNT,
  COUNT(DISTINCT ACCOUNT_ID) AS ACCOUNT_COUNT,
  MIN(OBSERVATION_DATE) AS FIRST_OBSERVATION,
  MAX(OBSERVATION_DATE) AS LAST_OBSERVATION,
  COUNT(DISTINCT OBSERVATION_DATE) AS OBSERVATION_MONTH_COUNT,
  COUNT(*) - COUNT(DISTINCT ACCOUNT_ID || '|' || OBSERVATION_DATE::VARCHAR) AS DUPLICATE_KEY_COUNT,
  COUNT_IF(OUTCOME_FINALITY_DATE > '2026-09-01'::DATE) AS NON_FINAL_LABEL_COUNT
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE;

In [ ]:
%%sql -r null_profile
SELECT
  COUNT_IF(ACCOUNT_ID IS NULL) AS ACCOUNT_ID_NULLS,
  COUNT_IF(OBSERVATION_DATE IS NULL) AS OBSERVATION_DATE_NULLS,
  COUNT_IF(PRODUCT_TYPE IS NULL) AS PRODUCT_TYPE_NULLS,
  COUNT_IF(ORIGINATION_CHANNEL IS NULL) AS ORIGINATION_CHANNEL_NULLS,
  COUNT_IF(TENURE_BAND IS NULL) AS TENURE_BAND_NULLS,
  COUNT_IF(ACCOUNT_AGE_MONTHS IS NULL) AS ACCOUNT_AGE_MONTHS_NULLS,
  COUNT_IF(CREDIT_LIMIT IS NULL) AS CREDIT_LIMIT_NULLS,
  COUNT_IF(UTILISATION_RATIO IS NULL) AS UTILISATION_RATIO_NULLS,
  COUNT_IF(MONTHLY_INCOME_ESTIMATE IS NULL) AS MONTHLY_INCOME_NULLS,
  COUNT_IF(PAYMENT_AMOUNT_30D IS NULL) AS PAYMENT_AMOUNT_NULLS,
  COUNT_IF(MISSED_PAYMENT_30D IS NULL) AS MISSED_PAYMENT_30D_NULLS,
  COUNT_IF(MISSED_PAYMENTS_6M IS NULL) AS MISSED_PAYMENTS_6M_NULLS,
  COUNT_IF(IN_ARREARS_30D IS NULL) AS IN_ARREARS_30D_NULLS,
  COUNT_IF(CUSTOMER_CONTACTS_90D IS NULL) AS CUSTOMER_CONTACTS_90D_NULLS,
  COUNT_IF(DEFAULT_WITHIN_90D IS NULL) AS TARGET_NULLS
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE;

A valid Phase 2 starting point has no duplicate account-date keys, no non-final outcomes in `TRAINING_BASE`, and no unexplained missing values. Any exception should stop modelling and be resolved in the data contract.

## 3. Understand target availability and time

Credit outcomes arrive after the 90-day window. The full outcome table therefore contains both finalised historical labels and pending recent observations. Training must use only the finalised relation.

In [ ]:
%%sql -r outcome_availability
SELECT
  OUTCOME_STATUS,
  COUNT(*) AS OUTCOME_COUNT,
  MIN(OBSERVATION_DATE) AS FIRST_OBSERVATION,
  MAX(OBSERVATION_DATE) AS LAST_OBSERVATION,
  AVG(IFF(
    OUTCOME_STATUS = 'FINALISED' AND OBSERVATION_DATE < '2026-01-01'::DATE,
    DEFAULT_WITHIN_90D,
    NULL
  )) AS PRE_HOLDOUT_DEFAULT_RATE
FROM CRISK_DEMO_DB.RAW.DEFAULT_OUTCOME
GROUP BY OUTCOME_STATUS
ORDER BY OUTCOME_STATUS;

In [ ]:
%%sql -r monthly_target
SELECT
  OBSERVATION_DATE,
  COUNT(*) AS TRAINING_ROW_COUNT,
  COUNT(DISTINCT ACCOUNT_ID) AS ACCOUNT_COUNT,
  SUM(DEFAULT_WITHIN_90D) AS DEFAULT_COUNT,
  AVG(DEFAULT_WITHIN_90D) AS DEFAULT_RATE
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
WHERE OBSERVATION_DATE < '2026-01-01'::DATE
GROUP BY OBSERVATION_DATE
ORDER BY OBSERVATION_DATE;

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

monthly_target_pdf = monthly_target.to_pandas()
monthly_target_pdf["OBSERVATION_DATE"] = pd.to_datetime(monthly_target_pdf["OBSERVATION_DATE"])

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
sns.lineplot(data=monthly_target_pdf, x="OBSERVATION_DATE", y="DEFAULT_RATE", marker="o", ax=axes[0])
axes[0].set_title("Pre-holdout 90-day default rate by observation month")
axes[0].set_ylabel("Default rate")

sns.barplot(data=monthly_target_pdf, x="OBSERVATION_DATE", y="TRAINING_ROW_COUNT", color="steelblue", ax=axes[1])
axes[1].set_title("Pre-holdout training observations by month")
axes[1].set_xlabel("Observation month")
axes[1].set_ylabel("Training rows")
axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

The pre-holdout monthly series reveals two modelling constraints: the target is imbalanced and observations are ordered in time. Random row splitting would place the same accounts and adjacent months on both sides of the split, producing optimistic evidence. Outcomes from 2026 onward remain untouched until final Phase 3 evaluation.

## 4. Discover candidate features from the observed schema

Classify the available columns only after inspecting the schema. This makes exclusions and candidate roles visible rather than embedding a preselected list inside training code.

In [ ]:
%%sql -r feature_catalog
SELECT
  COLUMN_NAME,
  DATA_TYPE,
  ORDINAL_POSITION,
  CASE
    WHEN COLUMN_NAME = 'ACCOUNT_ID' THEN 'IDENTIFIER_EXCLUDE'
    WHEN COLUMN_NAME IN ('OBSERVATION_DATE', 'OUTCOME_FINALITY_DATE') THEN 'TIME_CONTROL_EXCLUDE'
    WHEN COLUMN_NAME = 'DEFAULT_WITHIN_90D' THEN 'TARGET'
    WHEN DATA_TYPE ILIKE '%CHAR%' THEN 'CATEGORICAL_CANDIDATE'
    WHEN DATA_TYPE IN ('NUMBER', 'FLOAT', 'DOUBLE', 'REAL', 'DECIMAL', 'NUMERIC') THEN 'NUMERIC_CANDIDATE'
    ELSE 'REVIEW'
  END AS PROVISIONAL_ROLE
FROM CRISK_DEMO_DB.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'RAW'
  AND TABLE_NAME = 'TRAINING_BASE'
ORDER BY ORDINAL_POSITION;

The provisional roles are a review aid, not the final feature contract. Identifier and outcome-timing fields are excluded to prevent memorisation and leakage. Observation date controls temporal splitting but is not passed directly to the initial model.

## 5. Inspect numeric distributions

The following table exposes scale, centre, tails, and operational bounds before transformations are chosen.

In [ ]:
%%sql -r numeric_profile
SELECT 'ACCOUNT_AGE_MONTHS' AS FEATURE, MIN(ACCOUNT_AGE_MONTHS) AS MINIMUM, AVG(ACCOUNT_AGE_MONTHS) AS MEAN, MEDIAN(ACCOUNT_AGE_MONTHS) AS MEDIAN, MAX(ACCOUNT_AGE_MONTHS) AS MAXIMUM FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
UNION ALL
SELECT 'CREDIT_LIMIT', MIN(CREDIT_LIMIT), AVG(CREDIT_LIMIT), MEDIAN(CREDIT_LIMIT), MAX(CREDIT_LIMIT) FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
UNION ALL
SELECT 'UTILISATION_RATIO', MIN(UTILISATION_RATIO), AVG(UTILISATION_RATIO), MEDIAN(UTILISATION_RATIO), MAX(UTILISATION_RATIO) FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
UNION ALL
SELECT 'MONTHLY_INCOME_ESTIMATE', MIN(MONTHLY_INCOME_ESTIMATE), AVG(MONTHLY_INCOME_ESTIMATE), MEDIAN(MONTHLY_INCOME_ESTIMATE), MAX(MONTHLY_INCOME_ESTIMATE) FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
UNION ALL
SELECT 'PAYMENT_AMOUNT_30D', MIN(PAYMENT_AMOUNT_30D), AVG(PAYMENT_AMOUNT_30D), MEDIAN(PAYMENT_AMOUNT_30D), MAX(PAYMENT_AMOUNT_30D) FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
UNION ALL
SELECT 'MISSED_PAYMENTS_6M', MIN(MISSED_PAYMENTS_6M), AVG(MISSED_PAYMENTS_6M), MEDIAN(MISSED_PAYMENTS_6M), MAX(MISSED_PAYMENTS_6M) FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
UNION ALL
SELECT 'CUSTOMER_CONTACTS_90D', MIN(CUSTOMER_CONTACTS_90D), AVG(CUSTOMER_CONTACTS_90D), MEDIAN(CUSTOMER_CONTACTS_90D), MAX(CUSTOMER_CONTACTS_90D) FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
ORDER BY FEATURE;

In [ ]:
%%sql -r numeric_by_target
SELECT
  DEFAULT_WITHIN_90D,
  COUNT(*) AS OBSERVATION_COUNT,
  AVG(UTILISATION_RATIO) AS AVG_UTILISATION_RATIO,
  AVG(MONTHLY_INCOME_ESTIMATE) AS AVG_MONTHLY_INCOME,
  AVG(PAYMENT_AMOUNT_30D) AS AVG_PAYMENT_AMOUNT_30D,
  AVG(MISSED_PAYMENT_30D) AS MISSED_PAYMENT_RATE_30D,
  AVG(MISSED_PAYMENTS_6M) AS AVG_MISSED_PAYMENTS_6M,
  AVG(IN_ARREARS_30D) AS ARREARS_RATE_30D,
  AVG(CUSTOMER_CONTACTS_90D) AS AVG_CUSTOMER_CONTACTS_90D
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
WHERE OBSERVATION_DATE < '2026-01-01'::DATE
GROUP BY DEFAULT_WITHIN_90D
ORDER BY DEFAULT_WITHIN_90D;

In [ ]:
%%sql -r utilisation_distribution
SELECT
  FLOOR(UTILISATION_RATIO * 10) / 10 AS UTILISATION_BIN,
  DEFAULT_WITHIN_90D,
  COUNT(*) AS OBSERVATION_COUNT
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
WHERE OBSERVATION_DATE < '2026-01-01'::DATE
GROUP BY UTILISATION_BIN, DEFAULT_WITHIN_90D
ORDER BY UTILISATION_BIN, DEFAULT_WITHIN_90D;

In [ ]:
utilisation_pdf = utilisation_distribution.to_pandas()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=utilisation_pdf,
    x="UTILISATION_BIN",
    y="OBSERVATION_COUNT",
    hue="DEFAULT_WITHIN_90D",
    ax=ax,
)
ax.set_title("Utilisation distribution by finalised outcome")
ax.set_xlabel("Utilisation ratio bin")
ax.set_ylabel("Account-month observations")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

Differences between outcome classes indicate useful signal but do not establish causality. The logarithmic count axis keeps the minority default class visible. Scaling and transformation decisions remain deferred until the modelling pipeline is chosen.

## 6. Inspect categorical support and outcome rates

Operational segments can be modelling candidates and later monitoring slices. Small groups or unstable rates require caution even when aggregate performance is strong.

In [ ]:
%%sql -r product_outcomes
SELECT
  PRODUCT_TYPE,
  COUNT(*) AS OBSERVATION_COUNT,
  COUNT(DISTINCT ACCOUNT_ID) AS ACCOUNT_COUNT,
  SUM(DEFAULT_WITHIN_90D) AS DEFAULT_COUNT,
  AVG(DEFAULT_WITHIN_90D) AS DEFAULT_RATE
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
WHERE OBSERVATION_DATE < '2026-01-01'::DATE
GROUP BY PRODUCT_TYPE
ORDER BY DEFAULT_RATE DESC;

In [ ]:
%%sql -r channel_outcomes
SELECT
  ORIGINATION_CHANNEL,
  COUNT(*) AS OBSERVATION_COUNT,
  COUNT(DISTINCT ACCOUNT_ID) AS ACCOUNT_COUNT,
  SUM(DEFAULT_WITHIN_90D) AS DEFAULT_COUNT,
  AVG(DEFAULT_WITHIN_90D) AS DEFAULT_RATE
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
WHERE OBSERVATION_DATE < '2026-01-01'::DATE
GROUP BY ORIGINATION_CHANNEL
ORDER BY DEFAULT_RATE DESC;

In [ ]:
%%sql -r tenure_outcomes
SELECT
  TENURE_BAND,
  COUNT(*) AS OBSERVATION_COUNT,
  COUNT(DISTINCT ACCOUNT_ID) AS ACCOUNT_COUNT,
  SUM(DEFAULT_WITHIN_90D) AS DEFAULT_COUNT,
  AVG(DEFAULT_WITHIN_90D) AS DEFAULT_RATE
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
WHERE OBSERVATION_DATE < '2026-01-01'::DATE
GROUP BY TENURE_BAND
ORDER BY DEFAULT_RATE DESC;

In [ ]:
product_pdf = product_outcomes.to_pandas()
channel_pdf = channel_outcomes.to_pandas()
tenure_pdf = tenure_outcomes.to_pandas()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for data, category, title, axis in (
    (product_pdf, "PRODUCT_TYPE", "Product", axes[0]),
    (channel_pdf, "ORIGINATION_CHANNEL", "Origination channel", axes[1]),
    (tenure_pdf, "TENURE_BAND", "Tenure", axes[2]),
):
    sns.barplot(data=data, x=category, y="DEFAULT_RATE", ax=axis, color="steelblue")
    axis.set_title(f"Default rate by {title.lower()}")
    axis.set_xlabel(title)
    axis.set_ylabel("Finalised default rate")
    axis.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 7. Examine feature stability before the held-out period

A feature can be predictive yet operationally fragile. Compare development with the later validation period while leaving observations from 2026 onward untouched for final evaluation.

In [ ]:
%%sql -r scenario_profile
SELECT
  IFF(OBSERVATION_DATE >= '2025-07-01'::DATE, 'VALIDATION', 'DEVELOPMENT') AS ANALYSIS_PERIOD,
  COUNT(*) AS OBSERVATION_COUNT,
  AVG(DEFAULT_WITHIN_90D) AS DEFAULT_RATE,
  AVG(UTILISATION_RATIO) AS AVG_UTILISATION_RATIO,
  AVG(PAYMENT_AMOUNT_30D) AS AVG_PAYMENT_AMOUNT_30D,
  AVG(MISSED_PAYMENT_30D) AS MISSED_PAYMENT_RATE_30D,
  AVG(MISSED_PAYMENTS_6M) AS AVG_MISSED_PAYMENTS_6M,
  AVG(IN_ARREARS_30D) AS ARREARS_RATE_30D,
  AVG(CUSTOMER_CONTACTS_90D) AS AVG_CUSTOMER_CONTACTS_90D
FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
WHERE OBSERVATION_DATE < '2026-01-01'::DATE
GROUP BY ANALYSIS_PERIOD
ORDER BY ANALYSIS_PERIOD;

In [ ]:
scenario_pdf = scenario_profile.to_pandas()
scenario_long = scenario_pdf.melt(
    id_vars=["ANALYSIS_PERIOD"],
    value_vars=[
        "AVG_UTILISATION_RATIO",
        "MISSED_PAYMENT_RATE_30D",
        "AVG_MISSED_PAYMENTS_6M",
        "ARREARS_RATE_30D",
    ],
    var_name="MEASURE",
    value_name="VALUE",
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=scenario_long, x="MEASURE", y="VALUE", hue="ANALYSIS_PERIOD", ax=ax)
ax.set_title("Feature behaviour across development and validation")
ax.set_xlabel("Measure")
ax.set_ylabel("Mean or rate")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

Differences between development and validation provide an early stability check without exposing the held-out target. The controlled drift period begins in 2026 and remains reserved for Phase 3 final evaluation.

## 8. Define temporal development, validation, and held-out windows

Use contiguous observation periods. Development ends before validation, and the held-out test begins at the controlled drift boundary. No model or feature decision should use held-out outcomes.

In [ ]:
%%sql -r temporal_split_summary
WITH assigned AS (
  SELECT
    *,
    CASE
      WHEN OBSERVATION_DATE < '2025-07-01'::DATE THEN 'DEVELOPMENT'
      WHEN OBSERVATION_DATE < '2026-01-01'::DATE THEN 'VALIDATION'
      ELSE 'HELD_OUT_TEST'
    END AS SPLIT_NAME
  FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
)
SELECT
  SPLIT_NAME,
  MIN(OBSERVATION_DATE) AS FIRST_OBSERVATION,
  MAX(OBSERVATION_DATE) AS LAST_OBSERVATION,
  COUNT(*) AS OBSERVATION_COUNT,
  COUNT(DISTINCT ACCOUNT_ID) AS ACCOUNT_COUNT,
  IFF(SPLIT_NAME = 'HELD_OUT_TEST', NULL, SUM(DEFAULT_WITHIN_90D)) AS VISIBLE_DEFAULT_COUNT,
  IFF(SPLIT_NAME = 'HELD_OUT_TEST', NULL, AVG(DEFAULT_WITHIN_90D)) AS VISIBLE_DEFAULT_RATE
FROM assigned
GROUP BY SPLIT_NAME
ORDER BY FIRST_OBSERVATION;

The split is provisional until its row counts and target support are reviewed. Repeated accounts across periods are intentional for forward-looking evaluation, but no future row from an account can influence fitting for an earlier scoring date. Hyperparameter and threshold decisions use development and validation only.

## 9. Establish baselines and evaluation intent

Accuracy is not an informative primary metric for an imbalanced target. The operational baseline is the historical prevalence and the practical capacity constraint is reviewing the highest-risk 10% of scored accounts.

In [ ]:
%%sql -r baseline_summary
WITH assigned AS (
  SELECT
    DEFAULT_WITHIN_90D,
    CASE
      WHEN OBSERVATION_DATE < '2025-07-01'::DATE THEN 'DEVELOPMENT'
      WHEN OBSERVATION_DATE < '2026-01-01'::DATE THEN 'VALIDATION'
      ELSE 'HELD_OUT_TEST'
    END AS SPLIT_NAME
  FROM CRISK_DEMO_DB.RAW.TRAINING_BASE
)
SELECT
  SPLIT_NAME,
  COUNT(*) AS OBSERVATION_COUNT,
  IFF(SPLIT_NAME = 'HELD_OUT_TEST', NULL, AVG(DEFAULT_WITHIN_90D)) AS VISIBLE_PREVALENCE_BASELINE,
  0.10 AS TARGET_REVIEW_RATE,
  CEIL(COUNT(*) * 0.10) AS TARGET_REVIEW_COUNT
FROM assigned
GROUP BY SPLIT_NAME
ORDER BY SPLIT_NAME;

Phase 3 will compare candidates using:

- ROC AUC for ranking across thresholds.
- Average precision for minority-class ranking quality.
- Brier score and calibration evidence for probability quality.
- Recall among the highest-risk 10% of accounts for review-capacity relevance.
- Segment ROC AUC and support for product, origination channel, and tenure.

Configured candidate gates are ROC AUC ≥ 0.72, average precision ≥ 0.20, Brier score ≤ 0.16, and segment ROC AUC ≥ 0.62. These are demonstration thresholds and must not be treated as real credit-risk policy.

## 10. Provisional feature decision

**Include for the first candidate:**

- Categorical: `PRODUCT_TYPE`, `ORIGINATION_CHANNEL`, `TENURE_BAND`.
- Numeric: `ACCOUNT_AGE_MONTHS`, `CREDIT_LIMIT`, `UTILISATION_RATIO`, `MONTHLY_INCOME_ESTIMATE`, `PAYMENT_AMOUNT_30D`, `MISSED_PAYMENT_30D`, `MISSED_PAYMENTS_6M`, `IN_ARREARS_30D`, `CUSTOMER_CONTACTS_90D`.

**Exclude from model inputs:**

- `ACCOUNT_ID`: identifier and memorisation risk.
- `OBSERVATION_DATE`: split/control field; avoid learning a synthetic calendar shortcut in the first candidate.
- `OUTCOME_FINALITY_DATE`: label-availability information and direct leakage risk.
- `DEFAULT_WITHIN_90D`: target.

**Questions carried into Phase 3:**

- Do linear and tree-based candidates react differently to the scale and bounded counts?
- Does origination channel add stable signal or only segment variation?
- How much performance changes between validation and controlled-drift held-out data?
- Are probabilities sufficiently calibrated for prioritisation, or is post-fit calibration required?

This phase ends with a reviewable hypothesis. It does not train or register a model.